# WEST axisymmetric W transport

O-on-W sputtering → Boris transport (sheath, Coulomb collisions, thermal
force, cross-field diffusion) → ADAS W⁺..W¹⁰⁺ → PWI wall with
self-sputtering. Real WEST geometry, 2D axisymmetric (x=Z, y=R).

Run: `mpirun -np 4 spa_mac_mpi -in in.axi_west_emission -var Dperp 0.5`
(short test: `-var nwarm 20000 -var ndiag 5000 `).
Standalone plotting tools: `analysis/*.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection


def read_surf(fname):
    """Read a SPARTA 2D .surf file (Points / Lines sections).

    Returns (pts, lines): pts is an (npoint, 2) array of the raw file
    coordinates, lines an (nline, 2) array of 0-based point indices.
    In the axisymmetric layout the file columns are x = Z, y = R.
    """
    pts, lines, section = [], [], None
    with open(fname) as f:
        for raw in f:
            line = raw.split("#")[0].strip()
            if not line:
                continue
            if line == "Points":
                section = "points"
                continue
            if line == "Lines":
                section = "lines"
                continue
            tok = line.split()
            if section == "points" and len(tok) >= 3:
                pts.append((float(tok[1]), float(tok[2])))
            elif section == "lines" and len(tok) >= 3:
                lines.append((int(tok[1]) - 1, int(tok[2]) - 1))
    return np.array(pts), np.array(lines)


In [ ]:
pts, lines = read_surf("input/wall_fine.surf")
print(f"wall: {len(pts)} points, {len(lines)} lines")

# axi layout: file x = Z, file y = R -> plot as poloidal cross-section (R, Z)
Z, R = pts[:, 0], pts[:, 1]
segs = np.stack([np.column_stack([R[lines[:, 0]], Z[lines[:, 0]]]),
                 np.column_stack([R[lines[:, 1]], Z[lines[:, 1]]])], axis=1)

cZ, cR = read_surf("input/core.surf")[0].T          # file cols: x=Z, y=R
# close the loop for plotting
cZp = np.append(cZ, cZ[0]); cRp = np.append(cR, cR[0])

fig, ax = plt.subplots(figsize=(6, 9))
ax.add_collection(LineCollection(segs, colors="k", lw=1.0))
ax.plot(cRp, cZp, "-", color="C3", lw=1.5, label="core (psi_norm=0.1)")

# normal ticks: SPARTA n = (-dZ, dR) per segment (file coords), shown in (R,Z)
dZ = np.diff(cZp); dR = np.diff(cRp)
seglen = np.hypot(dZ, dR)
nZ, nR = -dR / seglen, dZ / seglen            # note: plotting (R,Z) so swap
mZ = 0.5 * (cZp[:-1] + cZp[1:]); mR = 0.5 * (cRp[:-1] + cRp[1:])
tick = 0.03
for k in range(0, len(mR), 12):               # every 12th segment
    ax.plot([mR[k], mR[k] + tick * nR[k]],
            [mZ[k], mZ[k] + tick * nZ[k]], "-", color="C0", lw=0.8)

ax.autoscale()
ax.set_aspect("equal")
ax.set_xlabel(r"$R$ (m)")
ax.set_ylabel(r"$Z$ (m)")
ax.set_title("WEST wall + core boundary (psi_norm=0.1)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, ls=":", alpha=0.4)
plt.show()

## Cross-field diffusion sweep — W density maps


In [ ]:
def read_grid_dump(fname):
    """Read a SPARTA `dump grid` text file; return {timestep: dict of columns}.

    Each frame's dict maps column name -> 1D numpy array over cells.
    """
    frames, cols, rows, ts = {}, None, [], None
    with open(fname) as f:
        it = iter(f)
        for line in it:
            if line.startswith("ITEM: TIMESTEP"):
                if ts is not None and rows:
                    dat = np.array(rows)
                    frames[ts] = {c: dat[:, k] for k, c in enumerate(cols)}
                ts = int(next(it))
                rows = []
            elif line.startswith("ITEM: CELLS"):
                cols = line.split()[2:]
            elif cols is not None and line[:5] != "ITEM:":
                tok = line.split()
                if len(tok) == len(cols):
                    rows.append([float(t) for t in tok])
    if ts is not None and rows:
        dat = np.array(rows)
        frames[ts] = {c: dat[:, k] for k, c in enumerate(cols)}
    return frames


import glob, re
# auto-detect which D_perp runs are present in state/ (one run or a sweep)
DPERP = sorted({re.search(r"grid\.dens\.west\.D(.+)$", f).group(1)
                for f in glob.glob("state/grid.dens.west.D*")},
               key=lambda s: float(s))
if not DPERP:
    raise FileNotFoundError("no state/grid.dens.west.D* dumps - run the deck first")
DTAG = DPERP[-1]                       # newest/only tag, for the single-run cells
print("D_perp runs found:", DPERP, "-> DTAG =", DTAG)

last = {}
for d in DPERP:
    frames = read_grid_dump(f"state/grid.dens.west.D{d}")
    ts = max(frames)
    g = last[d] = frames[ts]
    dens = g["f_fwgrid"]
    pos = dens[dens > 0]
    # axi cell volume = 2*pi*R_c * dZ * dR  (dump x slot = Z, y slot = R)
    vol = 2 * np.pi * g["yc"] * (g["xhi"] - g["xlo"]) * (g["yhi"] - g["ylo"])
    print(f"D_perp = {d:>4} m^2/s : step {ts}, {len(pos)} occupied cells, "
          f"p50 {np.percentile(pos, 50):.2e}  p99 {np.percentile(pos, 99):.2e}  "
          f"max {pos.max():.2e} m^-3,  total W inventory {(dens*vol).sum():.3e}")

In [ ]:
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm

# wall outline in (R, Z)
pts, wlines = read_surf("input/wall_fine.surf")
wZ, wR = pts[:, 0], pts[:, 1]
wall_segs = np.stack(
    [np.column_stack([wR[wlines[:, 0]], wZ[wlines[:, 0]]]),
     np.column_stack([wR[wlines[:, 1]], wZ[wlines[:, 1]]])], axis=1)

# One log scale shared by all three panels, set by pooled percentiles rather
# than the max: single heavy-weight particles parked in mm-scale refined
# divertor cells produce >1e16 m^-3 spikes (1-particle noise) that would
# otherwise wash out the whole map. Values above vmax saturate the ramp.
pooled = np.concatenate([last[d]["f_fwgrid"][last[d]["f_fwgrid"] > 0]
                         for d in DPERP])
vmin = np.percentile(pooled, 20.0)
vmax = np.percentile(pooled, 99.9)
norm = LogNorm(vmin=vmin, vmax=vmax)

fig, axes = plt.subplots(1, len(DPERP), figsize=(8*len(DPERP)+1, 7),
                         sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, d in zip(axes, DPERP):
    g = last[d]
    dens = g["f_fwgrid"]
    keep = dens > 0
    # cell rectangles: dump x = Z (axial slot), y = R  ->  plot (R, Z)
    quads = np.stack([
        np.column_stack([g["ylo"][keep], g["xlo"][keep]]),
        np.column_stack([g["yhi"][keep], g["xlo"][keep]]),
        np.column_stack([g["yhi"][keep], g["xhi"][keep]]),
        np.column_stack([g["ylo"][keep], g["xhi"][keep]])], axis=1)
    pc = PolyCollection(quads, array=np.clip(dens[keep], vmin, vmax),
                        cmap="viridis", norm=norm, edgecolors="none")
    ax.add_collection(pc)
    ax.add_collection(LineCollection(wall_segs, colors="0.25", lw=0.7))
    ax.set_xlim(1.75, 3.25)
    ax.set_ylim(-1.0, 0.85)
    ax.set_aspect("equal")
    ax.set_facecolor("0.94")          # zero-density cells show as background
    ax.set_xlabel(r"$R$ (m)")
    ax.set_title(rf"$D_\perp = {d}$ m$^2$/s")

    # ax.set_ylabel(r"$Z$ (m)")
    # cb = fig.colorbar(pc, ax=ax, shrink=0.85, pad=0.02)
    # cb.set_label(r"$\langle n_W \rangle$ (m$^{-3}$), all charge states"
    #              "\n(log scale clipped at pooled p99.9)")

axes[0].set_ylabel(r"$Z$ (m)")
cb = fig.colorbar(pc, ax=axes, shrink=0.85, pad=0.02)
cb.set_label(r"$\langle n_W \rangle$ (m$^{-3}$), all charge states"
         "\n(log scale clipped at pooled p99.9)")
fig.suptitle("WEST axi: time-averaged tungsten density vs cross-field diffusion",
             fontsize=12)
plt.show()

In [ ]:
# Impact of D_perp where it is physically expected: radial n_W profiles.
# Diffusion moves W ACROSS flux surfaces, so the profile perpendicular to
# the wall should broaden/flatten with D_perp while the D = 0 case stays
# pinned to the field lines threading the source region.
#
# Two cuts: outboard midplane (|Z| < 0.10 m, LFS approach R -> outer wall)
# and a horizontal cut through the lower-divertor source region
# (Z in [-0.75, -0.55]). Volume-weighted cell averages, log y.

cuts = [("outboard midplane, |Z| < 0.10", -0.10, 0.10),
        ("lower divertor band, Z in [-0.75, -0.55]", -0.75, -0.55)]
colors = {"0": "#4053d3", "0.1": "#00b25d", "1": "#ddb310", "10.0": "#b51d14"}

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for ax, (label, zlo, zhi) in zip(axs, cuts):
    for d in DPERP:
        g = last[d]
        sel = (g["xc"] > zlo) & (g["xc"] < zhi)
        R = g["yc"][sel]
        n = g["f_fwgrid"][sel]
        vol = g["yc"][sel] * (g["xhi"] - g["xlo"])[sel] * (g["yhi"] - g["ylo"])[sel]
        bins = np.linspace(1.8, 3.25, 60)
        num = np.histogram(R, bins, weights=n * vol)[0]
        den = np.histogram(R, bins, weights=vol)[0]
        prof = np.where(den > 0, num / np.maximum(den, 1e-300), np.nan)
        ax.semilogy(0.5 * (bins[1:] + bins[:-1]), prof,
                    color=colors.get(d, "k"), lw=1.8,
                    label=rf"$D_\perp = {d}$")
    ax.set_xlabel(r"$R$ (m)")
    ax.set_title(label, fontsize=10)
    ax.grid(True, ls=":", alpha=0.4)
axs[0].set_ylabel(r"$\langle n_W \rangle$ (m$^{-3}$)")
axs[0].legend(frameon=False, fontsize=9)
fig.suptitle("Radial W density profiles vs cross-field diffusion", fontsize=12)
plt.show()

In [ ]:

from matplotlib.path import Path

pts, wlines = read_surf("input/wall_fine.surf")

# order the closed loop by adjacency
adj = {}
for a, b in wlines:
    adj.setdefault(a, []).append(b)
    adj.setdefault(b, []).append(a)
loop = [wlines[0][0], wlines[0][1]]
while True:
    nxt = [k for k in adj[loop[-1]] if k != loop[-2]]
    if not nxt or nxt[0] == loop[0]:
        break
    loop.append(nxt[0])
P = pts[loop]                       # (N, 2) ordered points, cols = (Z, R)

# start at max-R point (outer midplane / antenna), go toward +Z (ceiling)
i0 = np.argmax(P[:, 1])
P = np.roll(P, -i0, axis=0)
if P[1, 0] < P[0, 0]:               # next point moves down in Z -> reverse
    P = np.roll(P[::-1], 1, axis=0)

seg = np.diff(np.vstack([P, P[:1]]), axis=0)        # (N, 2) element vectors
slen = np.sqrt((seg ** 2).sum(1))
s_mid = np.concatenate([[0], np.cumsum(slen)])[:-1] + 0.5 * slen
mid = P + 0.5 * seg

# inward unit normal via point-in-polygon test at +eps
poly = Path(P)
eps = 0.01
nrm = np.column_stack([-seg[:, 1], seg[:, 0]]) / slen[:, None]
inside = poly.contains_points(mid + eps * nrm)
nrm[~inside] *= -1.0

def wall_density(g, q):
    """Mean cell density at query points q (cols Z, R) from grid frame g."""
    out = np.zeros(len(q))
    for k, (zq, rq) in enumerate(q):
        m = ((g["xlo"] <= zq) & (g["xhi"] > zq) &
             (g["ylo"] <= rq) & (g["yhi"] > rq))
        if m.any():
            out[k] = g["f_fwgrid"][m].mean()
    return out

qpts = mid + eps * nrm
sbins = np.arange(0.0, s_mid.max() + 0.05, 0.05)
s_ctr = 0.5 * (sbins[1:] + sbins[:-1])

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
for d in DPERP:
    nw = wall_density(last[d], qpts)
    num = np.histogram(s_mid, sbins, weights=nw * slen)[0]
    den = np.histogram(s_mid, sbins, weights=slen)[0]
    prof = np.where(den > 0, num / np.maximum(den, 1e-300), np.nan)
    ax.semilogy(s_ctr, np.where(prof > 0, prof, np.nan),
                color=colors.get(d, "k"), lw=1.5, label=rf"$D_\perp = {d}$")

# PFC landmarks from geometry extremes
marks = [("antenna", 0.0),
         ("ceiling", s_mid[np.argmax(mid[:, 0])]),
         ("inner wall", s_mid[np.argmin(mid[:, 1])]),
         ("lower divertor", s_mid[np.argmin(mid[:, 0])])]
for label, sm in marks:
    ax.axvline(sm, color="0.75", lw=0.8, ls="--")
    ax.annotate(label, (sm, 0.03), xycoords=("data", "axes fraction"),
                ha="center", va="bottom", fontsize=8, color="0.35",
                rotation=90)

ax.set_xlabel(r"wall coordinate $s$ (m)")
ax.set_ylabel(r"$\langle n_W \rangle$ 1 cm inside wall (m$^{-3}$)")
ax.grid(True, ls=":", alpha=0.4)
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.set_title("Near-wall W density along the wall",
             fontsize=11)
plt.show()

In [ ]:
def read_surf_dump(fname):
    frames, cols, rows, ts = {}, None, [], None
    it = iter(open(fname))
    for line in it:
        if line.startswith("ITEM: TIMESTEP"):
            if ts is not None and rows:
                d = np.array(rows); frames[ts] = {c: d[:, k] for k, c in enumerate(cols)}
            ts = int(next(it)); rows = []
        elif line.startswith("ITEM: SURFS"):
            cols = line.split()[2:]
        elif cols is not None and not line.startswith("ITEM:"):
            t = line.split()
            if len(t) == len(cols): rows.append([float(x) for x in t])
    if rows: d = np.array(rows); frames[ts] = {c: d[:, k] for k, c in enumerate(cols)}
    return frames[max(frames)]

def wall_s_mapping(fx):
    """Map each flux element to arc length s along the ordered wall loop
    (s=0 at the outer midplane, walking so Z increases first)."""
    eZ = 0.5 * (fx["v1x"] + fx["v2x"]); eR = 0.5 * (fx["v1y"] + fx["v2y"])
    eL = np.hypot(fx["v2x"] - fx["v1x"], fx["v2y"] - fx["v1y"])
    pw, lw = read_surf("input/wall_fine.surf")
    adj = {}
    for a, b in lw:
        adj.setdefault(a, []).append(b); adj.setdefault(b, []).append(a)
    lp = [lw[0][0], lw[0][1]]
    while True:
        nx = [k for k in adj[lp[-1]] if k != lp[-2]]
        if not nx or nx[0] == lp[0]: break
        lp.append(nx[0])
    P = pw[lp]; P = np.roll(P, -np.argmax(P[:, 1]), axis=0)
    if P[1, 0] < P[0, 0]: P = np.roll(P[::-1], 1, axis=0)
    sL = np.hypot(*np.diff(np.vstack([P, P[:1]]), axis=0).T)
    s_pt = np.concatenate([[0], np.cumsum(sL)])[:-1]
    from scipy.spatial import cKDTree
    s_elem = s_pt[cKDTree(P).query(np.column_stack([eZ, eR]))[1]]
    return s_elem, eL, s_pt, P

DTAG = DPERP[-1]
fx = read_surf_dump(f"state/wall.flux.west.D{DTAG}")
ero  = fx["c_cero[1]" if "c_cero[1]" in fx else "c_cero"]
dep  = fx["f_fdep[1]"]
load = fx["f_fdep[2]"]
s_elem, eL, s_pt, P = wall_s_mapping(fx)

sb = np.arange(0, s_pt.max() + 0.05, 0.05); sc = 0.5 * (sb[1:] + sb[:-1])
def prof(v):
    num = np.histogram(s_elem, sb, weights=v * eL)[0]
    den = np.histogram(s_elem, sb, weights=eL)[0]
    return np.where(den > 0, num / np.maximum(den, 1e-300), np.nan)
pe, pd, pl = prof(ero), prof(dep), prof(load)

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.semilogy(sc, np.where(pe > 0, pe, np.nan), "-",  color="C3", lw=1.8, label="erosion (gross O$\\to$W sputter)")
ax.semilogy(sc, np.where(pd > 0, pd, np.nan), "-",  color="C0", lw=1.8, label="deposition (redeposited W)")
ax.semilogy(sc, np.where(pl > 0, pl, np.nan), "--", color="0.5", lw=1.2, label="gross wall load (all incidence)")
for lab, sm in [("antenna", 0.0), ("ceiling", s_pt[np.argmax(P[:, 0])]),
                ("inner wall", s_pt[np.argmin(P[:, 1])]), ("lower divertor", s_pt[np.argmin(P[:, 0])])]:
    ax.axvline(sm, color="0.85", lw=0.8, ls="--")
    ax.annotate(lab, (sm, 0.02), xycoords=("data", "axes fraction"),
                ha="center", va="bottom", fontsize=8, color="0.4", rotation=90)
ax.set_xlabel(r"wall coordinate $s$ (m)"); ax.set_ylabel(r"W flux (m$^{-2}$ s$^{-1}$)")
ax.set_title(f"W erosion, deposition & load vs wall coordinate (D={DTAG})")
ax.legend(frameon=False, fontsize=9); ax.grid(True, ls=":", alpha=0.4)
plt.show()
print(f"peak erosion {np.nanmax(pe):.2e}, peak deposition {np.nanmax(pd):.2e}, "
      f"peak load {np.nanmax(pl):.2e} m^-2 s^-1")


In [ ]:

import h5py

DTAG = DPERP[-1]
g = read_grid_dump(f"state/grid.dens.west.D{DTAG}"); g = g[max(g)]
Z, R, n = g["xc"], g["yc"], g["f_fwgrid"]
vol = 2 * np.pi * R * (g["xhi"] - g["xlo"]) * (g["yhi"] - g["ylo"])   # axi m^3

# psi_norm at each cell from the equilibrium map
with h5py.File("input/plasma.h5", "r") as f:
    psi = f["equilibrium/psi"][:]; rg = f["equilibrium/r"][:]; zg = f["equilibrium/z"][:]
    pax = float(f["equilibrium/psi_axis"][()]); pb = float(f["equilibrium/psib"][()])
pn_map = (psi - pax) / (pb - pax)                    # psi indexed [iz, ir]
def psi_norm_at(R, Z):
    ir = np.clip(np.searchsorted(rg, R) - 1, 0, len(rg) - 2)
    iz = np.clip(np.searchsorted(zg, Z) - 1, 0, len(zg) - 2)
    tr = (R - rg[ir]) / (rg[ir+1] - rg[ir]); tz = (Z - zg[iz]) / (zg[iz+1] - zg[iz])
    return ((1-tr)*(1-tz)*pn_map[iz, ir] + tr*(1-tz)*pn_map[iz, ir+1]
            + (1-tr)*tz*pn_map[iz+1, ir] + tr*tz*pn_map[iz+1, ir+1])
pn = psi_norm_at(R, Z)

# gross erosion rate phi_W (rate column; falls back to single c_cero)
fx = read_surf_dump(f"state/wall.flux.west.D{DTAG}")
rate_col = "c_cero[2]" if "c_cero[2]" in fx else "c_cero"
phiW = fx[rate_col].sum()
Nin = (n[pn < 1.0] * vol[pn < 1.0]).sum()
tauW = Nin / phiW
band = (pn > 0.10) & (pn < 0.25) & (n > 0)
nWbound = n[band].mean() if band.any() else 0.0

print(f"n_W^bound (psi_norm 0.10-0.25) = {nWbound:.3e} m^-3 ")
print(f"N_in (W inside separatrix)     = {Nin:.3e} particles")
print(f"phi_W (gross erosion rate)     = {phiW:.3e} /s   [{rate_col}]")
print(f"tau_W = N_in/phi_W             = {tauW:.3e} s   log10 = {np.log10(tauW):+.2f}"
      f"   (log10 tau_W ~ -3 .. -8)")
print(f"W content: confined (psi_n<1) {100*(n*vol)[pn<1.0].sum()/(n*vol).sum():.1f}%"
      f"  SOL (psi_n>1) {100*(n*vol)[pn>=1.0].sum()/(n*vol).sum():.1f}%")

# n_W(psi_norm) penetration profile: volume-weighted mean density per psi bin
pbins = np.linspace(0, 2.0, 41); pc = 0.5 * (pbins[1:] + pbins[:-1])
num = np.histogram(pn[n > 0], pbins, weights=(n * vol)[n > 0])[0]
den = np.histogram(pn[n > 0], pbins, weights=vol[n > 0])[0]
prof = np.where(den > 0, num / np.maximum(den, 1e-300), np.nan)
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.semilogy(pc, np.where(prof > 0, prof, np.nan), "o-", color="C2", ms=4)
ax.axvline(1.0, color="0.5", ls="--", lw=1); ax.text(1.02, ax.get_ylim()[1]*0.3, "separatrix", fontsize=8, color="0.4")
ax.axvline(0.1, color="C3", ls="--", lw=1); ax.text(0.12, ax.get_ylim()[1]*0.3, "core bnd", fontsize=8, color="C3")
ax.set_xlabel(r"$\psi_{norm}$  (0=axis, 1=separatrix)")
ax.set_ylabel(r"$\langle n_W \rangle$ (m$^{-3}$)")
ax.set_title(r"W penetration profile $n_W(\psi_{norm})$ — SOL (right) into confined (left)")
ax.grid(True, ls=":", alpha=0.4)
plt.show()

In [ ]:
# Ionization-length diagnostic: neutral-W density vs distance-to-wall.
# W neutrals fly ballistically from the wall and ionize over the ionization
# mean free path lambda_ion, so n_W0(d) ~ exp(-d/lambda_ion). A short decay
# (~cm) where the plasma is dense (divertor) confirms neutrals ionize near
# the wall; a long tail is the physical ballistic flight in the thin SOL.
# Needs state/ndens0.west.D<D> from a run WITH the neutral tally (added to
# in.axi_west_emission during the ionization-length discussion) — re-run to
# generate it.
import os
DTAG = DPERP[-1]   # match the -var Dperp of the run
fn = f"state/ndens0.west.D{DTAG}"
if not os.path.exists(fn):
    print(f"{fn} not found — re-run in.axi_west_emission to produce the "
          "neutral-W tally (dump dw0).")
else:
    g = read_grid_dump(fn); g = g[max(g)]
    dist = g["c_cgeom[1]"]        # per-cell distance to nearest wall [m]
    n0   = g["f_fw0"]             # time-averaged neutral-W density [m^-3]
    sel  = n0 > 0

    fig, axs = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)

    # (a) neutral-W density map
    sc = axs[0].scatter(g["yc"][sel], g["xc"][sel], c=n0[sel], s=6,
                        cmap="magma", norm=LogNorm())
    axs[0].add_collection(LineCollection(segs, colors="0.5", lw=0.5))
    axs[0].set_xlim(1.75, 3.25); axs[0].set_ylim(-1.0, 0.85)
    axs[0].set_aspect("equal")
    axs[0].set_xlabel(r"$R$ (m)"); axs[0].set_ylabel(r"$Z$ (m)")
    axs[0].set_title(r"neutral-W density (hugs wall = short $\lambda_{ion}$)")
    fig.colorbar(sc, ax=axs[0], shrink=0.8, label=r"$n_{W^0}$ (m$^{-3}$)")

    # (b) n_W0 vs distance-to-wall — the decay length is lambda_ion.
    bins = np.linspace(0, 0.20, 41)
    ctr  = 0.5 * (bins[1:] + bins[:-1])
    num  = np.histogram(dist[sel], bins, weights=n0[sel])[0]
    cnt  = np.histogram(dist[sel], bins)[0]
    prof = np.where(cnt > 0, num / np.maximum(cnt, 1), np.nan)
    axs[1].semilogy(ctr * 100, prof, "o-", color="C3", ms=4)
    axs[1].set_xlabel("distance to wall (cm)")
    axs[1].set_ylabel(r"$\langle n_{W^0} \rangle$ (m$^{-3}$)")
    axs[1].set_title(r"neutral decay $\sim e^{-d/\lambda_{ion}}$")
    axs[1].grid(True, ls=":", alpha=0.4)
    good = np.isfinite(prof) & (prof > 0)
    if good.sum() > 3:
        slope = np.polyfit(ctr[good], np.log(prof[good]), 1)[0]
        if slope < 0:
            axs[1].text(0.5, 0.9, rf"$\lambda_{{ion}} \approx {-1/slope*100:.1f}$ cm",
                        transform=axs[1].transAxes, fontsize=11)
    plt.show()

In [ ]:
DTAG = DPERP[-1]
fx = read_surf_dump(f"state/wall.flux.west.D{DTAG}")
ero = fx["c_cero[1]" if "c_cero[1]" in fx else "c_cero"]
dep = fx["f_fdep[1]"]
neterode = ero - dep
s_elem, eL, s_pt, P = wall_s_mapping(fx)
eR = 0.5 * (fx["v1y"] + fx["v2y"])
ring = 2 * np.pi * np.maximum(eR, 0) * eL          # axi ring area [m^2]

sb = np.arange(0, s_pt.max() + 0.05, 0.05); sc = 0.5 * (sb[1:] + sb[:-1])
num = np.histogram(s_elem, sb, weights=neterode * eL)[0]
den = np.histogram(s_elem, sb, weights=eL)[0]
netp = np.where(den > 0, num / np.maximum(den, 1e-300), np.nan)

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.semilogy(sc, np.where(netp > 0, netp, np.nan), "-o", color="C3", ms=3, label="net EROSION (source)")
ax.semilogy(sc, np.where(netp < 0, -netp, np.nan), "-o", color="C0", ms=3, label="net DEPOSITION (sink)")
for lab, sm in [("antenna", 0.0), ("ceiling", s_pt[np.argmax(P[:, 0])]),
                ("inner wall", s_pt[np.argmin(P[:, 1])]), ("lower divertor", s_pt[np.argmin(P[:, 0])])]:
    ax.axvline(sm, color="0.85", lw=0.8, ls="--")
    ax.annotate(lab, (sm, 0.02), xycoords=("data", "axes fraction"),
                ha="center", fontsize=8, color="0.4", rotation=90)
ax.set_xlabel(r"wall coordinate $s$ (m)"); ax.set_ylabel(r"$|$net W flux$|$ (m$^{-2}$ s$^{-1}$)")
ax.set_title(f"Net W erosion vs wall coordinate (D={DTAG}) — red erodes, blue accumulates")
ax.legend(frameon=False, fontsize=9); ax.grid(True, ls=":", alpha=0.4)
plt.show()

E = (ero * ring).sum(); Dp = (dep * ring).sum(); L = (fx["f_fdep[2]"] * ring).sum()
print(f"gross erosion rate   E = {E:.3e} /s")
print(f"deposition rate      D = {Dp:.3e} /s   (redeposition fraction D/E = {Dp/E*100:.1f}%)")
print(f"net erosion rate   E-D = {E-Dp:.3e} /s")
print(f"gross wall load      L = {L:.3e} /s   (L/D = {L/Dp:.2f})")


In [ ]:
# --- Convergence: total W atoms vs time (from the stats log) ------------
# Uses the pweight-weighted, mesh-independent atom counts printed by the
# updated deck: stats_style ... c_cwNtot (total) c_cwN0..c_cwN10 (per charge
# state). Needs a run with that deck; a plateau in c_cwNtot => steady state.
import glob

def read_log_stats(fname):
    lines = open(fname).read().splitlines()
    blocks, i = [], 0
    while i < len(lines):
        if lines[i].split()[:1] == ["Step"]:
            hdr = lines[i].split(); data = []; j = i + 1
            while j < len(lines):
                p = lines[j].split()
                if len(p) != len(hdr): break
                try: data.append([float(x) for x in p])
                except ValueError: break
                j += 1
            if data: blocks.append((hdr, np.array(data)))
            i = j
        else:
            i += 1
    return blocks

CS = ["W0", "W+"] + [f"W{q}+" for q in range(2, 11)]
logs = [f for f in glob.glob("log*") if "c_cwNtot" in open(f).read()]
if not logs:
    print("No log with convergence columns yet. Rerun in.axi_west_emission with "
          "the updated deck, e.g.:\n"
          "  mpirun -np 4 ~/build_oe/src/spa_mac_mpi -in in.axi_west_emission "
          "-var Dperp 0.5 -log log.d05")
else:
    blocks = read_log_stats(sorted(logs)[-1])
    # warmup block, then diagnostic block (step resets at reset_timestep 0);
    # lay them on one continuous axis and mark the warmup|diagnostic boundary.
    xs, tot, spec, off, reset_x = [], [], {q: [] for q in range(11)}, 0.0, None
    for bi, (hdr, d) in enumerate(blocks):
        x = d[:, hdr.index("Step")] + off
        xs.append(x); tot.append(d[:, hdr.index("c_cwNtot")])
        for q in range(11):
            spec[q].append(d[:, hdr.index(f"c_cwN{q}")])
        if bi == 0: reset_x = x[-1]
        off = x[-1]
    x = np.concatenate(xs); tot = np.concatenate(tot)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.3), constrained_layout=True)
    a1.plot(x, tot, "-", color="k", lw=1.8)
    if reset_x: a1.axvline(reset_x, color="C1", ls="--", lw=1, label="warmup | diagnostic")
    a1.set_xlabel("cumulative step"); a1.set_ylabel(r"total W atoms  ($\Sigma$ pweight)")
    a1.set_title("Total W vs time — convergence"); a1.legend(frameon=False, fontsize=8)
    a1.grid(True, ls=":", alpha=0.4)
    for q in range(11):
        yq = np.concatenate(spec[q])
        if np.nanmax(yq) > 0:
            a2.semilogy(x, np.where(yq > 0, yq, np.nan), lw=1.2, label=CS[q])
    if reset_x: a2.axvline(reset_x, color="C1", ls="--", lw=1)
    a2.set_xlabel("cumulative step"); a2.set_ylabel("W atoms per charge state")
    a2.set_title("Charge-state populations vs time")
    a2.legend(frameon=False, fontsize=7, ncol=2); a2.grid(True, ls=":", alpha=0.4)
    plt.show()

    tail = tot[-max(len(tot) // 10, 1):]
    drift = (tot[-1] - tail.mean()) / tot[-1]
    print(f"final total W = {tot[-1]:.3e} atoms;  last-decile drift = {drift*100:+.1f}%  "
          f"(|drift| < ~1% => converged)")


In [ ]:

import glob, os
Ub = 8.68                                   # W surface binding energy [eV]
# W_on_W Eckstein params (eckstein_sputter_data.h): Eth, Q, ETF
Eth, Q, ETF = 62.06, 33.47, 1998893.0       # W_on_W (eckstein_sputter_data.h)

def bohdansky_Y(E):
    E = np.asarray(E, float)
    y = np.zeros_like(E)
    m = E > Eth
    x = Eth / E[m]
    eps = E[m] / ETF
    sn = 0.5*np.log(1+1.2288*eps) / (eps + 0.1728*np.sqrt(eps) + 0.008*eps**0.1504)
    y[m] = Q*(1-x**(2/3))*(1-x)**2 * sn
    return np.clip(y, 0, None)

def read_ave_vector(fname, nbin):
    rows = [l.split() for l in open(fname) if l.strip() and not l.startswith("#")]
    vals = [float(b) for a, b in rows if a.isdigit()]
    return np.array(vals[-nbin:]) if len(vals) >= nbin else None

NB_IMP = 48
imp_edges = np.logspace(0, np.log10(3000), NB_IMP + 1)
imp_ctr = np.sqrt(imp_edges[1:] * imp_edges[:-1])
fimp = f"state/impact.hist.west.D{DTAG}"
if os.path.exists(fimp):
    w_in = read_ave_vector(fimp, NB_IMP)
    src = f"convolved over measured impact spectrum (D={DTAG})"
else:
    w_in = np.exp(-(np.log(imp_ctr) - np.log(112))**2 / 0.5)   # fallback ~112 eV
    src = "impact.hist not found - using representative <E_in>=112 eV"

Eg = np.logspace(-1, 3.2, 400)
spec = np.zeros_like(Eg)
for Ein, w in zip(imp_ctr, w_in * bohdansky_Y(imp_ctr)):
    Emax = Ein - Ub                          # gamma ~ 1
    if w <= 0 or Emax <= 0:
        continue
    f = np.where(Eg <= Emax,
                 Eg/(Eg+Ub)**3 * (1 - np.sqrt(np.clip((Eg+Ub)/(Emax+Ub), 0, 1))), 0.0)
    spec += w * f
if spec.max() > 0:
    spec /= np.trapz(spec, Eg)               # normalize to a PDF

fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
ax.semilogx(Eg, spec, "-", color="C1", lw=2, label="sputtered W (Thompson + cutoff)")
ax.axvline(Ub/2, color="0.6", ls=":", lw=1)
ax.annotate(f"peak ~Ub/2={Ub/2:.1f} eV", (Ub/2, ax.get_ylim()[1]*0.9),
            fontsize=8, color="0.4", rotation=90, va="top")
Emean = np.trapz(Eg*spec, Eg) if spec.max() > 0 else 0
ax.set_xlabel("sputtered energy E (eV)"); ax.set_ylabel("f(E) (1/eV, normalized)")
ax.set_title(f"Sputtered-W energy spectrum \n{src}")
ax.legend(frameon=False, fontsize=9); ax.grid(True, ls=":", alpha=0.4)
plt.show()
print(f"mean sputtered energy <E> = {Emean:.1f} eV  (Thompson tail, recoil-capped)")


## 2D total-W density map (all charge states, final frame)


In [ ]:
# --- 2D total-W density map from the consolidated grid dump ---
import numpy as np, matplotlib.pyplot as plt
D = DTAG if 'DTAG' in dir() else '0.1'
fn = f'state/grid.dens.west.D{D}'
ls = open(fn).readlines()
i0 = [i for i, l in enumerate(ls) if l.startswith('ITEM: TIMESTEP')][-1]
n = int(ls[i0+3]); cols = ls[i0+8].split()[2:]
d = np.array([l.split() for l in ls[i0+9:i0+9+n]], float)
G = {c: d[:, k] for k, c in enumerate(cols)}

# box-deposit cell values onto uniform bins (axi: x = Z, y = R)
NX, NY = 370, 325
xe = np.linspace(-1.0, 0.85, NX+1); ye = np.linspace(0.0, 3.25, NY+1)
area = (G['xhi']-G['xlo']) * (G['yhi']-G['ylo'])
Hn, _, _ = np.histogram2d(G['xc'], G['yc'], bins=[xe, ye],
                          weights=G['f_fwgrid']*area)
Ha, _, _ = np.histogram2d(G['xc'], G['yc'], bins=[xe, ye], weights=area)
nW = np.where(Ha > 0, Hn/np.maximum(Ha, 1e-30), np.nan)

# wall polyline for overlay
rz, on = [], False
for l in open('input/wall_fine.surf'):
    t = l.split()
    if not t: continue
    if t[0] == 'Points': on = True; continue
    if on:
        if len(t) == 3: rz.append((float(t[1]), float(t[2])))
        elif rz: break
rz = np.array(rz)

vmax = np.nanmax(nW)
fig, ax = plt.subplots(figsize=(7.5, 9), dpi=130)
pc = ax.pcolormesh(xe, ye, nW.T, cmap='magma',
                   norm=plt.matplotlib.colors.LogNorm(vmin=vmax*1e-4, vmax=vmax))
plt.colorbar(pc, ax=ax, label=r'$n_W$ (all charge states) [m$^{-3}$]')
ax.plot(rz[:, 0], rz[:, 1], 'c-', lw=1.0)
ax.set_xlabel('Z [m]'); ax.set_ylabel('R [m]'); ax.set_aspect('equal')
ax.set_title(f'total W density, $D_\\perp$ = {D}')
fig.tight_layout()
fig.savefig(f'state/nW_total_D{D}.png', dpi=200, bbox_inches='tight')
plt.show()
